# Dan 2 — Full fine-tuning na GPU-u (Kaggle / Colab)

Lokalno (CPU, Ryzen 7 5700U) treniramo samo **glavu** nad zamrznutim embeddinzima —
jedna epoha punog fine-tuninga trajala bi ~95 min, 15 epoha ≈ 24h. Zato pun
fine-tuning ide ovde, na besplatnom GPU-u.

**Pre pokretanja:**
- **Kaggle:** desni panel → *Session options* → Accelerator **GPU T4 x2** ili **P100**,
  Internet **On**. (Oboje traži verifikovan telefon: Settings → Phone Verification.)
- **Colab:** *Runtime → Change runtime type → T4 GPU*.

Notebook je namerno tanak — sva logika je u repou (`dogbreeds/`), pa je isti kod
testiran lokalno i pokrenut na GPU-u. Nema kopiranja koda između okruženja.

## 0. Okruženje i kod

In [ ]:
import os, sys, glob, json, subprocess
from pathlib import Path

REPO_URL = "https://github.com/banep33/kp-dog-breeds.git"

ON_KAGGLE = os.path.exists('/kaggle/working')
ON_COLAB  = 'google.colab' in sys.modules

if ON_KAGGLE:
    WORK, repo = Path('/kaggle/working'), Path('/kaggle/working/kp-dog-breeds')
elif ON_COLAB:
    WORK, repo = Path('/content/work'), Path('/content/kp-dog-breeds')
else:
    WORK, repo = Path('work'), Path('.')
WORK.mkdir(parents=True, exist_ok=True)

if ON_KAGGLE or ON_COLAB:
    if repo.exists():
        subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=False)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(repo)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'timm>=1.0.15', 'optuna', 'wandb', 'cleanlab',
                    'onnxruntime'], check=True)
os.chdir(repo); sys.path.insert(0, str(repo))


def sh(cmd: str, check: bool = True) -> int:
    """Pokreni komandu i PADNI ako nije uspela.

    Zasto ne `!python ...`: nenulti exit kod shell komande ne prekida celiju, pa
    Kaggle prijavi "Successful" i kad je korak pukao. Tako je 14.10. ONNX izvoz
    tiho otkazao (nedostajao onnxruntime), a verzija je bila zelena."""
    print('$', cmd, flush=True)
    code = subprocess.run(cmd, shell=True).returncode
    if code != 0:
        if check:
            raise RuntimeError(f'Komanda pukla (exit {code}): {cmd}')
        print(f'[upozorenje] exit {code}, nastavljam: {cmd}', flush=True)
    return code


import torch
print('repo  :', repo)
print('torch :', torch.__version__)
print('CUDA  :', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'Nema GPU-a! Ukljuci akcelerator u Session options.'

## 1. Podaci

Dva izvora, isti format (`labels.csv` sa kolonama `id,breed` + folder `train/<id>.jpg`):

| Izvor | Slika | Treba nalog? |
|---|---|---|
| Kaggle *Dog Breed Identification* | 10.222 | da — prihvaćena pravila takmičenja |
| Stanford Dogs | 20.580 | ne |

Ćelija ispod **sama nalazi** šta je zakačeno kao Input (Kaggle je menjao strukturu
foldera), pa putanje ne pogađamo.

In [ ]:
DATA_DIR = None; LABELS_CSV = None

if ON_KAGGLE:
    for d in sorted(glob.glob('/kaggle/input/*')):
        print('input:', d)
    found    = sorted(glob.glob('/kaggle/input/**/labels.csv', recursive=True))
    stanford = sorted(glob.glob('/kaggle/input/**/n02085620-Chihuahua', recursive=True))
    if found:
        LABELS_CSV = Path(found[0]); DATA_DIR = LABELS_CSV.parent
        print('-> Kaggle takmicenje:', LABELS_CSV)
    elif stanford:
        from scripts.stanford_to_kaggle import convert
        DATA_DIR = Path('/tmp/data'); DATA_DIR.mkdir(parents=True, exist_ok=True)
        LABELS_CSV = convert(Path(stanford[0]).parent, DATA_DIR, copy=True)
        print('-> Stanford Dogs dataset:', LABELS_CSV)
    else:
        raise RuntimeError('Nema podataka. Desni panel -> Input -> + Add Input -> '
                           'takmicenje dog-breed-identification ili dataset Stanford Dogs.')
else:
    from scripts.stanford_to_kaggle import download_and_convert
    DATA_DIR = Path('/content/data') if ON_COLAB else Path('data/raw')
    LABELS_CSV = download_and_convert(DATA_DIR)

IMAGES_DIR = Path(DATA_DIR) / 'train'
import pandas as pd
_df = pd.read_csv(LABELS_CSV)
print(f'{len(_df)} slika, {_df.breed.nunique()} rasa, slike u {IMAGES_DIR}')

## 2. Deduplikacija + split

Isti kod kao lokalno. Na GPU-u je ekstrakcija embeddinga ~30× brža (minut-dva
umesto 25 minuta).

**Zašto grupni split:** dataset sadrži serije fotografija istog psa. Bez grupisanja
skoro-identične slike završe sa obe strane train/test granice i tačnost je lažno
visoka. Prag 0,93 je izabran iz sweep tabele (vidi `reports/dedup.json`).

In [ ]:
EMB     = WORK / 'emb' / 'convnext_tiny'
GROUPED = WORK / 'labels_grouped.csv'
SPLITS  = WORK / 'splits'

sh(f'python -m dogbreeds.embed --labels {LABELS_CSV} --images {IMAGES_DIR} '
   f'--model convnext_tiny.dinov3_lvd1689m --out {EMB} --batch-size 128 --num-workers 2')

sh(f'python -m dogbreeds.dedup --emb {EMB} --labels {LABELS_CSV} '
   f'--threshold 0.93 --out {GROUPED} --report {WORK}/dedup.json')

In [ ]:
sh(f'python -m dogbreeds.data --config configs/kaggle.yaml --set '
   f'data.labels_csv={GROUPED} data.splits_dir={SPLITS}')

# provera: nijedna grupa ne sme biti u dva skupa
parts = {s: pd.read_csv(SPLITS / f'{s}.csv') for s in ['train','val','test']}
g = {s: set(d['group']) for s, d in parts.items()}
assert not (g['train'] & g['val']) and not (g['train'] & g['test']) \
       and not (g['val'] & g['test']), 'CURENJE GRUPA!'
print('OK - nema curenja grupa:', {s: len(d) for s, d in parts.items()})

## 3. Baseline za poređenje — linear probe

Pokrećemo ga i ovde da bi brojevi bili merljivi sa lokalnima. Fine-tuning ispod
**mora** da pobedi ovaj broj — inače nema razloga da ga radimo.

In [ ]:
sh(f'python -m dogbreeds.probe --emb {EMB} --splits {SPLITS} '
   f'--out {WORK}/runs/probe_convnext --report {WORK}/probe_results.json')

## 4. Praćenje eksperimenata (W&B) — opciono

Napravi besplatan nalog na [wandb.ai](https://wandb.ai), pa **Kaggle:** desni panel →
*Add-ons → Secrets* → dodaj `WANDB_API_KEY`. Bez ovoga trening radi normalno, samo
nema grafika u oblaku (istorija ostaje u `history.csv`).

In [ ]:
USE_WANDB = False
try:
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
    if os.environ.get('WANDB_API_KEY'):
        import wandb; wandb.login(); USE_WANDB = True
except Exception as e:
    print('W&B preskocen:', e)
print('W&B:', USE_WANDB)

## 5. Full fine-tuning

Recept (svaku stavku treba umeti objasniti — `docs/INTERVIEW.md`):

| Tehnika | Zašto |
|---|---|
| zamrznut backbone 1. epohu | nasumična glava bi velikim gradijentima pokvarila dobre pretrenirane težine |
| discriminative LR (`backbone_lr_mult=0.1`) | backbone već zna da gleda; glava uči od nule |
| AdamW + cosine + warmup | stabilan start, glatko spuštanje LR-a |
| label smoothing 0.1 | 120 klasa, neke se stvarno preklapaju — ne teramo preteranu sigurnost |
| RandAugment + Random Erasing | slike korisnika su raznovrsnije od dataseta |
| EMA težina | prosek težina kroz vreme generalizuje bolje od poslednjeg koraka |
| AMP (fp16) | ~2× brže na T4, bez gubitka tačnosti |

Počni sa `efficientnet_b0` (najbrži, pa je iteracija jeftina).

In [ ]:
MODEL = 'efficientnet_b0.ra4_e3600_r224_in1k'
RUN   = 'b0_ft'

sh(f'python -m dogbreeds.train --config configs/kaggle.yaml --set '
   f'data.splits_dir={SPLITS} data.images_dir={IMAGES_DIR} '
   f'model.name={MODEL} train.epochs=15 '
   f'output.dir={WORK}/runs output.run_name={RUN} '
   f'logging.wandb={str(USE_WANDB).lower()}')

## 6. Evaluacija + kalibracija

`evaluate.py` fituje temperaturu na **val** skupu i primenjuje je na **test** — nikad
obrnuto. Temperatura ne menja top-1/top-3 (redosled klasa ostaje isti), nego popravlja
*verovatnoće* koje prikazujemo korisniku.

In [ ]:
CKPT = f'{WORK}/runs/{RUN}/best.pt'
sh(f'python -m dogbreeds.evaluate --checkpoint {CKPT} --config configs/kaggle.yaml --set '
   f'data.splits_dir={SPLITS} data.images_dir={IMAGES_DIR}')

m = json.load(open(f'{WORK}/runs/{RUN}/eval/metrics.json'))
print(json.dumps(m, indent=2))

## 7. Izvoz za produkciju (ONNX + INT8)

ONNX oslobađa model od PyTorch-a (Docker ~200 MB umesto ~2 GB), a INT8 kvantizacija
dodatno smanjuje fajl i ubrzava CPU. **Obavezno** posle kvantizacije ponovo izmeriti
tačnost — brzina bez tačnosti ne vredi ništa.

In [ ]:
ART = f'{WORK}/artifacts'
sh(f'python -m dogbreeds.export --checkpoint {CKPT} --out {ART} --int8 '
   f'--config configs/kaggle.yaml --set '
   f'data.splits_dir={SPLITS} data.images_dir={IMAGES_DIR}')

sh(f'python scripts/eval_onnx.py {ART}/model.onnx --splits {SPLITS} --images {IMAGES_DIR}')

# INT8 je poboljsanje, ne uslov: FP32 vec staje u budzet (p95 20,6 ms < 100 ms,
# 16,6 MB < 50 MB). Zato check=False -- ako kvantizacija ne uspe, isporucujemo FP32.
sh(f'python scripts/eval_onnx.py {ART}/model.int8.onnx --splits {SPLITS} --images {IMAGES_DIR}',
   check=False)

## 8. Preuzmi rezultate

**Kaggle:** desni panel → *Output* → preuzmi `artifacts/` (`model.onnx`, `model.json`)
i `runs/<run>/` (`history.csv`, `best_metrics.json`, `eval/`). Pouzdanije: **Save Version**
(*Save & Run All*) — prolaz ide na serveru, preživljava zatvaranje taba, a output ostaje
trajno zakačen za verziju.

**Colab:** fajlovi su u `/content/work` — prebaci na Drive ili preuzmi.

Lokalno onda:
```powershell
python -m dogbreeds.inference artifacts\model.onnx moj_pas.jpg
```

> **Ne koristi `model.int8.onnx`.** INT8 kvantizacija na ovoj arhitekturi daje model na
> nivou nasumičnog pogađanja — uzrok je izmeren i zapisan u `reports/int8_diagnosis.json`
> (40% tenzora ima negativan SQNR). Zato `export.py` od sada proverava kandidata protiv
> FP32 reference i **briše ga ako se ne slaže**. Isporučuje se FP32.


In [ ]:
import shutil
print('--- artifacts ---')
for p in sorted(Path(ART).glob('*')):
    print(f'  {p.name:<22}{p.stat().st_size/1e6:7.1f} MB')
print('--- run ---')
for p in sorted(Path(f'{WORK}/runs/{RUN}').rglob('*')):
    if p.is_file():
        print('  ', p.relative_to(WORK))

if ON_KAGGLE:
    shutil.make_archive('/kaggle/working/rezultati', 'zip', root_dir=str(WORK))
    print('zip: /kaggle/working/rezultati.zip')